# Data Preparation for ASP Processing

This notebook prepares raw satellite images before running the Ames Stereo Pipeline (ASP).

The objective is to create a clean working folder with short and consistent filenames. Instead of using long original product names, the selected image tiles and their metadata are copied, converted, or mosaicked into:

```bash
merged_tiles/
```

This folder is then used in the next ASP steps.

For a tri-stereo acquisition, the expected output is:

```bash
merged_tiles/
├── A.tif
├── A.XML
├── DIM_A.XML
├── B.tif
├── B.XML
├── DIM_B.XML
├── C.tif
├── C.XML
└── DIM_C.XML
```

For a stereo pair, the same logic is used with only two images:

```bash
merged_tiles/
├── A.tif
├── A.XML
├── DIM_A.XML
├── B.tif
├── B.XML
└── DIM_B.XML
```

The original raw data are not modified. This notebook only creates a simplified working copy for the analysis.

The user must define the input structure in the configuration section of the script, including:

- `Site_Name` and `Date`;
- `base_dir` and `output_dir`;
- `Platform`, for example `PHR1A`, `PHR1B`, `PNEO4`, `SPOT6`, or `SPOT7`;
- `ACQUISITION_MODE`, either `stereo` or `tri_stereo`;
- `MERGE_TILES`, either `no` for one tile or `yes` for several tiles;
- `Tile_IDs`, for example `R1C1`, `R2C1`, or both;
- `input_folders`, which must match the real product folder structure;
- accepted image extensions, for example `.TIF`, `.tif`, `.JP2`, or `.jp2`.

The same script can therefore be adapted to Pléiades, Pléiades Neo, SPOT, or another ASP-compatible optical stereo dataset, as long as the correct folders and metadata patterns are provided.

In [26]:
%%bash

# ============================================================
# Data preparation for ASP
#
# Purpose:
#   Prepare stereo or tri-stereo satellite images using short names:
#   A.tif, B.tif, C.tif
#
# Supported cases:
#   1) Stereo pair, one tile per acquisition
#   2) Tri-stereo, one tile per acquisition
#   3) Stereo or tri-stereo, several tiles merged per acquisition
#
# The raw data are not modified.
# Prepared files are written to output_dir.
# ============================================================

set -e

# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# 1.1 Site and date
# ------------------------------------------------------------
# These names must match the real folder names on disk.
# Example:
#   Site_Name="Berared"
#   Date="Aug24"
# ------------------------------------------------------------

Site_Name="Berared"
Date="Aug24"


# ------------------------------------------------------------
# 1.2 Base input folder and output folder
# ------------------------------------------------------------
# Edit these two paths depending on the sensor/data archive.
#
# Example Pléiades:
# base_dir="/mnt/summer/USERS/KOAI/Pleiades_DINAMIS/${Site_Name}/${Date}"
# output_dir="/mnt/summer/USERS/KOAI/${Site_Name}_${Date}/merged_tiles"
#
# Example Pléiades Neo:
# base_dir="/mnt/summer/USERS/KOAI/Pleiades_NEO/${Site_Name}/${Date}"
# output_dir="/mnt/summer/USERS/KOAI/Pleiades_Neo_Analysis/${Site_Name}_${Date}/merged_tiles"
#
# Example SPOT:
# base_dir="/mnt/summer/USERS/KOAI/SPOT_DINAMIS/${Site_Name}/${Date}"
# output_dir="/mnt/summer/USERS/KOAI/SPOT_Analysis/${Site_Name}_${Date}/merged_tiles"
# ------------------------------------------------------------

base_dir="/mnt/summer/USERS/KOAI/Pleiades_DINAMIS/${Site_Name}/${Date}"
output_dir="/mnt/summer/USERS/KOAI/${Site_Name}_${Date}/merged_tiles"


# ------------------------------------------------------------
# 1.3 Platform name
# ------------------------------------------------------------
# Examples:
#   Pléiades:      Platform="PHR1A" or Platform="PHR1B"
#   Pléiades Neo:  Platform="PNEO3" or Platform="PNEO4"
#   SPOT:          Platform="SPOT6" or Platform="SPOT7"
#
# The script searches for:
#   RPC_${Platform}*.XML
#   DIM_${Platform}*.XML
# ------------------------------------------------------------

Platform="PHR1A"


# ------------------------------------------------------------
# 1.4 Acquisition mode
# ------------------------------------------------------------
# Options:
#   ACQUISITION_MODE="stereo"      -> output A, B
#   ACQUISITION_MODE="tri_stereo"  -> output A, B, C
# ------------------------------------------------------------

ACQUISITION_MODE="tri_stereo"


# ------------------------------------------------------------
# 1.5 Tile mode
# ------------------------------------------------------------
# Options:
#   MERGE_TILES="no"   -> use one selected tile only
#   MERGE_TILES="yes"  -> merge several selected tiles into one image
# ------------------------------------------------------------

MERGE_TILES="no"


# ------------------------------------------------------------
# 1.6 Selected tile or tiles
# ------------------------------------------------------------
# One tile:
#   Tile_IDs=("R1C1")
#
# Multiple tiles:
#   Tile_IDs=("R1C1" "R2C1")
# ------------------------------------------------------------

Tile_IDs=("R1C1")


# ------------------------------------------------------------
# 1.7 Accepted image extensions
# ------------------------------------------------------------
# Keep all unless the product uses only one known format.
# ------------------------------------------------------------

Image_Extensions=("TIF" "tif" "JP2" "jp2")


# ------------------------------------------------------------
# 1.8 Input folders relative to base_dir
# ------------------------------------------------------------
# The number of folders must match the acquisition mode:
#   stereo      -> 2 folders
#   tri_stereo  -> 3 folders
#
# Example Pléiades tri-stereo:
# input_folders=(
#     "Pleiades_1/IMG_${Platform}_P_001"
#     "Pleiades_2/IMG_${Platform}_P_001"
#     "Pleiades_3/IMG_${Platform}_P_001"
# )
#
# Example Pléiades Neo stereo:
# input_folders=(
#     "Pleiades_1/IMG_01_${Platform}_PAN"
#     "Pleiades_2/IMG_02_${Platform}_PAN"
# )
#
# Example SPOT stereo:
# input_folders=(
#     "SPOT_1_20240717_PAN/IMG_${Platform}_P_001_A"
#     "SPOT_2_20240724_PAN/IMG_${Platform}_P_001_A"
# )
# ------------------------------------------------------------

input_folders=(
    "Pleiades_1/IMG_${Platform}_P_001"
    "Pleiades_2/IMG_${Platform}_P_001"
    "Pleiades_3/IMG_${Platform}_P_001"
)


# ------------------------------------------------------------
# 1.9 GDAL commands
# ------------------------------------------------------------
# /usr/bin versions are used to avoid possible conflicts with
# ASP internal GDAL libraries.
# ------------------------------------------------------------

GDAL_TRANSLATE="/usr/bin/gdal_translate"
GDAL_BUILDVRT="/usr/bin/gdalbuildvrt"


# ============================================================
# 2. AUTOMATIC SETTINGS
# ============================================================

if [ "$ACQUISITION_MODE" = "stereo" ]; then
    short_names=("A" "B")
elif [ "$ACQUISITION_MODE" = "tri_stereo" ]; then
    short_names=("A" "B" "C")
else
    echo "ERROR: ACQUISITION_MODE must be 'stereo' or 'tri_stereo'."
    exit 1
fi


# ============================================================
# 3. BASIC CHECKS
# ============================================================

echo "============================================================"
echo "DATA PREPARATION SETTINGS"
echo "============================================================"
echo "Site name        : $Site_Name"
echo "Date             : $Date"
echo "Acquisition mode : $ACQUISITION_MODE"
echo "Merge tiles      : $MERGE_TILES"
echo "Platform         : $Platform"
echo "Base directory   : $base_dir"
echo "Output directory : $output_dir"
echo "Tile IDs         : ${Tile_IDs[*]}"
echo "Image extensions : ${Image_Extensions[*]}"
echo "Short names      : ${short_names[*]}"
echo "Input folders    : ${input_folders[*]}"
echo "============================================================"
echo ""

if [ ! -d "$base_dir" ]; then
    echo "ERROR: base_dir does not exist:"
    echo "$base_dir"
    exit 1
fi

mkdir -p "$output_dir"

if [ "${#input_folders[@]}" -ne "${#short_names[@]}" ]; then
    echo "ERROR: Number of input folders does not match acquisition mode."
    echo "Expected number : ${#short_names[@]}"
    echo "Provided folders: ${#input_folders[@]}"
    exit 1
fi

if [ "$MERGE_TILES" = "no" ] && [ "${#Tile_IDs[@]}" -ne 1 ]; then
    echo "ERROR: MERGE_TILES='no' requires exactly one Tile_ID."
    exit 1
fi

if [ "$MERGE_TILES" != "yes" ] && [ "$MERGE_TILES" != "no" ]; then
    echo "ERROR: MERGE_TILES must be 'yes' or 'no'."
    exit 1
fi


# ============================================================
# 4. PROCESS EACH ACQUISITION
# ============================================================

for idx in "${!input_folders[@]}"; do

    short_name="${short_names[$idx]}"
    rel_folder="${input_folders[$idx]}"
    folder="${base_dir}/${rel_folder}"

    echo ""
    echo "============================================================"
    echo "Preparing acquisition: $short_name"
    echo "Input folder: $folder"
    echo "============================================================"

    if [ ! -d "$folder" ]; then
        echo "ERROR: Cannot find input folder:"
        echo "$folder"
        exit 1
    fi

    cd "$folder" || exit 1


    # --------------------------------------------------------
    # 4.1 Find RPC and DIM metadata
    # --------------------------------------------------------

    rpc_file=$(ls RPC_${Platform}*.XML RPC_${Platform}*.xml 2>/dev/null | head -n 1 || true)
    dim_file=$(ls DIM_${Platform}*.XML DIM_${Platform}*.xml 2>/dev/null | head -n 1 || true)

    if [ -z "$rpc_file" ]; then
        echo "ERROR: No RPC file found with pattern:"
        echo "RPC_${Platform}*.XML"
        echo "Available XML files:"
        ls -lh *.XML *.xml 2>/dev/null || true
        exit 1
    fi

    if [ -z "$dim_file" ]; then
        echo "WARNING: No DIM file found with pattern DIM_${Platform}*.XML"
    fi

    echo "RPC file: $rpc_file"
    if [ -n "$dim_file" ]; then
        echo "DIM file: $dim_file"
    fi


    # --------------------------------------------------------
    # 4.2 Find selected image tile(s)
    # --------------------------------------------------------

    tile_files=()

    for tile in "${Tile_IDs[@]}"; do

        img_file=""

        for ext in "${Image_Extensions[@]}"; do
            candidate=$(ls *"${tile}.${ext}" 2>/dev/null | head -n 1 || true)

            if [ -n "$candidate" ]; then
                img_file="$candidate"
                break
            fi
        done

        if [ -z "$img_file" ]; then
            echo "WARNING: No image found for tile: $tile"
        else
            echo "Found tile $tile: $img_file"
            tile_files+=("$img_file")
        fi

    done

    if [ "${#tile_files[@]}" -eq 0 ]; then
        echo "ERROR: No selected image tiles found for acquisition $short_name."
        echo "Available image files:"
        ls -lh *.TIF *.tif *.JP2 *.jp2 2>/dev/null || true
        exit 1
    fi


    # --------------------------------------------------------
    # 4.3 Create simplified GeoTIFF
    # --------------------------------------------------------

    out_tif="${output_dir}/${short_name}.tif"
    out_vrt="${output_dir}/${short_name}.vrt"

    if [ "$MERGE_TILES" = "no" ]; then

        img_file="${tile_files[0]}"

        echo "Using one tile only: $img_file"

        case "$img_file" in
            *.TIF|*.tif)
                echo "Copying TIF -> $out_tif"
                cp "$img_file" "$out_tif"
                ;;

            *.JP2|*.jp2)
                echo "Converting JP2 -> $out_tif"
                env -u GDAL_DRIVER_PATH -u LD_LIBRARY_PATH \
                    "$GDAL_TRANSLATE" "$img_file" "$out_tif" -co TILED=YES
                ;;

            *)
                echo "ERROR: Unsupported image extension:"
                echo "$img_file"
                exit 1
                ;;
        esac

    elif [ "$MERGE_TILES" = "yes" ]; then

        if [ "${#tile_files[@]}" -eq 1 ]; then

            img_file="${tile_files[0]}"

            echo "Only one selected tile was found. No mosaic needed."

            case "$img_file" in
                *.TIF|*.tif)
                    echo "Copying TIF -> $out_tif"
                    cp "$img_file" "$out_tif"
                    ;;

                *.JP2|*.jp2)
                    echo "Converting JP2 -> $out_tif"
                    env -u GDAL_DRIVER_PATH -u LD_LIBRARY_PATH \
                        "$GDAL_TRANSLATE" "$img_file" "$out_tif" -co TILED=YES
                    ;;

                *)
                    echo "ERROR: Unsupported image extension:"
                    echo "$img_file"
                    exit 1
                    ;;
            esac

        else

            echo "Merging selected tiles:"
            printf '  %s\n' "${tile_files[@]}"

            env -u GDAL_DRIVER_PATH -u LD_LIBRARY_PATH \
                "$GDAL_BUILDVRT" "$out_vrt" "${tile_files[@]}"

            env -u GDAL_DRIVER_PATH -u LD_LIBRARY_PATH \
                "$GDAL_TRANSLATE" "$out_vrt" "$out_tif" -co TILED=YES

        fi
    fi


    # --------------------------------------------------------
    # 4.4 Copy metadata using short names
    # --------------------------------------------------------

    cp "$rpc_file" "${output_dir}/${short_name}.XML"

    if [ -n "$dim_file" ]; then
        cp "$dim_file" "${output_dir}/DIM_${short_name}.XML"
    fi

    echo ""
    echo "Created:"
    echo "  ${output_dir}/${short_name}.tif"
    echo "  ${output_dir}/${short_name}.XML"

    if [ -n "$dim_file" ]; then
        echo "  ${output_dir}/DIM_${short_name}.XML"
    fi

done


# ============================================================
# 5. FINAL CHECK
# ============================================================

echo ""
echo "============================================================"
echo "Final content of merged_tiles:"
echo "============================================================"

ls -lh "$output_dir"

echo ""
echo "Data preparation completed successfully."

DATA PREPARATION SETTINGS
Site name        : Berared
Date             : Aug24
Acquisition mode : tri_stereo
Merge tiles      : no
Platform         : PHR1A
Base directory   : /mnt/summer/USERS/KOAI/Pleiades_DINAMIS/Berared/Aug24
Output directory : /mnt/summer/USERS/KOAI/Berared_Aug24/merged_tiles
Tile IDs         : R1C1
Image extensions : TIF tif JP2 jp2
Short names      : A B C
Input folders    : Pleiades_1/IMG_PHR1A_P_001 Pleiades_2/IMG_PHR1A_P_001 Pleiades_3/IMG_PHR1A_P_001


Preparing acquisition: A
Input folder: /mnt/summer/USERS/KOAI/Pleiades_DINAMIS/Berared/Aug24/Pleiades_1/IMG_PHR1A_P_001
RPC file: RPC_PHR1A_P_202408301041244_SEN_7101262101-1.XML
DIM file: DIM_PHR1A_P_202408301041244_SEN_7101262101-1.XML
Found tile R1C1: IMG_PHR1A_P_202408301041244_SEN_7101262101-1_R1C1.TIF
Using one tile only: IMG_PHR1A_P_202408301041244_SEN_7101262101-1_R1C1.TIF
Copying TIF -> /mnt/summer/USERS/KOAI/Berared_Aug24/merged_tiles/A.tif

Created:
  /mnt/summer/USERS/KOAI/Berared_Aug24/merged_tiles/

## Metadata, Overlap, and Stereo-Geometry Check

This step checks the prepared `merged_tiles/` folder before starting ASP processing.

The objective is to verify that the prepared images and metadata are ready for the next ASP steps. The block generates and displays three clean tables:

1. **Overlap pairs**: valid stereo pairs and approximate image-overlap values.
2. **Image metadata**: image-level information extracted from the DIM files, displayed with parameters as rows and images as columns.
3. **Stereo geometry**: pair-level geometry, including B/H ratio and stereo angle.

This block uses the combined script stored in the same folder as this notebook:

```bash
prepare_stereo_metadata_and_geometry.py
```

The script reads the prepared image/RPC/DIM files from:

```bash
merged_tiles/
```

and creates temporary CSV outputs for:

```bash
overlap_pairs
image_metadata
stereo_geometry
```

By default, `SAVE_OUTPUTS = False`, so the CSV files are only temporary and the results are displayed in the notebook. Set `SAVE_OUTPUTS = True` if the tables should be saved permanently in the `merged_tiles/` folder.

The B/H ratio and stereo angle are computed from the image geometry metadata, using the incidence along-track angle, incidence across-track angle, and azimuth angle extracted from the DIM files.

In [25]:
import subprocess
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


# ============================================================
# 1. USER SETTINGS
# ============================================================

Site_Name = "Berarde"
Date = "Jul24"

SAVE_OUTPUTS = False
IOU_THRESHOLD = "0.05"


# ============================================================
# 2. SELECT PREPARED DATA FOLDER
# ============================================================
# Keep the option you want active.
# Comment the others using #.

# ------------------------------------------------------------
# Option 1: Pléiades
# Example expected files:
# A.tif, A.XML, DIM_A.XML
# B.tif, B.XML, DIM_B.XML
# C.tif, C.XML, DIM_C.XML
# ------------------------------------------------------------
# work_dir = Path(f"/mnt/summer/USERS/KOAI/{Site_Name}_{Date}/merged_tiles")
# pair_args = []


# ------------------------------------------------------------
# Option 2: Pléiades Neo
# Example expected files:
# A.tif, A.XML, DIM_A.XML
# B.tif, B.XML, DIM_B.XML
# ------------------------------------------------------------
# work_dir = Path(f"/mnt/summer/USERS/KOAI/Pleiades_Neo_Analysis/{Site_Name}_{Date}/merged_tiles")
# pair_args = []


# ------------------------------------------------------------
# Option 3: SPOT, normal A/B case
# Example expected files:
# A.tif, A.XML, DIM_A.XML
# B.tif, B.XML, DIM_B.XML
# ------------------------------------------------------------
work_dir = Path(f"/mnt/summer/USERS/KOAI/SPOT_Analysis/{Site_Name}_{Date}/merged_tiles")
pair_args = []


# ------------------------------------------------------------
# Option 4: SPOT, multiple custom stereo pairs
# Example expected files:
# A_1.tif, A_1.XML, DIM_A_1.XML
# B_1.tif, B_1.XML, DIM_B_1.XML
# A_2.tif, A_2.XML, DIM_A_2.XML
# B_2.tif, B_2.XML, DIM_B_2.XML
# ------------------------------------------------------------
# work_dir = Path(f"/mnt/summer/USERS/KOAI/SPOT_Analysis/{Site_Name}_{Date}/merged_tiles")
# pair_args = [
#     "--pairs",
#     "A_1:B_1",
#     "A_2:B_2",
# ]


# ============================================================
# 3. SCRIPT LOCATION
# ============================================================
# The script is stored in the same folder as this notebook.

script_path = Path("prepare_stereo_metadata_and_geometry.py")


# ============================================================
# 4. BASIC CHECKS
# ============================================================

print("Site name:", Site_Name)
print("Date:", Date)
print("Working directory:", work_dir)
print("Script:", script_path)
print("Save outputs:", SAVE_OUTPUTS)

if not work_dir.exists():
    raise FileNotFoundError(f"Working directory not found:\n{work_dir}")

if not script_path.exists():
    raise FileNotFoundError(f"Script not found:\n{script_path}")


# ============================================================
# 5. RUN METADATA / OVERLAP / GEOMETRY SCRIPT
# ============================================================

if SAVE_OUTPUTS:
    out_prefix = work_dir / Site_Name
    temp_dir = None
else:
    temp_dir = tempfile.TemporaryDirectory()
    out_prefix = Path(temp_dir.name) / Site_Name


cmd = [
    "python",
    str(script_path),
    "-img_dir",
    str(work_dir),
    "-out_prefix",
    str(out_prefix),
    "-thresh",
    IOU_THRESHOLD,
] + pair_args


print("\nRunning command:")
print(" ".join(str(c) for c in cmd))

subprocess.run(cmd, check=True)


# ============================================================
# 6. READ OUTPUT TABLES
# ============================================================

overlap_csv = Path(f"{out_prefix}_overlap_pairs.csv")
metadata_csv = Path(f"{out_prefix}_image_metadata.csv")
geometry_csv = Path(f"{out_prefix}_stereo_geometry.csv")

overlap_df = pd.read_csv(overlap_csv)
metadata_df = pd.read_csv(metadata_csv)
geometry_df = pd.read_csv(geometry_csv)


# ============================================================
# 7. ORGANIZE IMAGE METADATA TABLE
# ============================================================
# The original metadata table has images as rows.
# This converts it to:
#   Parameter | A | B | C
# which is easier for comparing acquisition geometry.

if "image_id" not in metadata_df.columns:
    raise ValueError("metadata_df must contain an 'image_id' column.")

metadata_vertical_df = (
    metadata_df
    .set_index("image_id")
    .T
    .reset_index()
    .rename(columns={"index": "Parameter"})
)

# Optional: move the most important parameters to the top
preferred_order = [
    "view_order",
    "tif_file",
    "rpc_file",
    "dim_file",
    "IMAGING_DATE",
    "IMAGING_TIME",
    "NBANDS",
    "NBITS",
    "FOCAL_LENGTH",
    "AZIMUTH_ANGLE",
    "VIEWING_ANGLE_ACROSS_TRACK",
    "VIEWING_ANGLE_ALONG_TRACK",
    "VIEWING_ANGLE",
    "INCIDENCE_ANGLE_ALONG_TRACK",
    "INCIDENCE_ANGLE_ACROSS_TRACK",
    "INCIDENCE_ANGLE",
    "SUN_AZIMUTH",
    "SUN_ELEVATION",
]

existing_preferred = [
    p for p in preferred_order
    if p in metadata_vertical_df["Parameter"].values
]

remaining_parameters = [
    p for p in metadata_vertical_df["Parameter"].values
    if p not in existing_preferred
]

ordered_parameters = existing_preferred + remaining_parameters

metadata_vertical_df = (
    metadata_vertical_df
    .set_index("Parameter")
    .loc[ordered_parameters]
    .reset_index()
)


# ============================================================
# 8. DISPLAY CLEAN TABLES
# ============================================================

display(Markdown("## Overlap pairs"))
display(overlap_df)

display(Markdown("## Image metadata"))
display(metadata_vertical_df)

display(Markdown("## Stereo geometry: B/H ratio and stereo angle"))
display(geometry_df)


# ============================================================
# 9. FINAL MESSAGE
# ============================================================

if SAVE_OUTPUTS:
    print("\nSaved output files:")
    print(overlap_csv)
    print(metadata_csv)
    print(geometry_csv)
else:
    print("\nSAVE_OUTPUTS = False, so CSV files were temporary and only displayed.")

if temp_dir is not None:
    temp_dir.cleanup()

Site name: Berarde
Date: Jul24
Working directory: /mnt/summer/USERS/KOAI/SPOT_Analysis/Berarde_Jul24/merged_tiles
Script: prepare_stereo_metadata_and_geometry.py
Save outputs: False

Running command:
python prepare_stereo_metadata_and_geometry.py -img_dir /mnt/summer/USERS/KOAI/SPOT_Analysis/Berarde_Jul24/merged_tiles -out_prefix /tmp/tmpuhh8vxmg/Berarde -thresh 0.05

Detected prepared image IDs:
['A', 'B']

Stereo pairs to evaluate:
  - A / B

Saved:
/tmp/tmpuhh8vxmg/Berarde_overlap_pairs.csv
/tmp/tmpuhh8vxmg/Berarde_image_metadata.csv
/tmp/tmpuhh8vxmg/Berarde_stereo_geometry.csv


## Overlap pairs

,pair,left_image,right_image,iou,is_valid,note
0,AB,A,B,1.0,True,NaN


## Image metadata

image_id,Parameter,A,B
0,view_order,Image_1,Image_2
1,tif_file,A.tif,B.tif
2,rpc_file,A.XML,B.XML
3,dim_file,DIM_A.XML,DIM_B.XML
4,IMAGING_DATE,2024-07-17,2024-07-24
5,IMAGING_TIME,10:03:05.5,09:59:09.4
6,NBANDS,1,1
7,NBITS,16,16
8,FOCAL_LENGTH,3.76036,3.76036
9,AZIMUTH_ANGLE,182.328488,116.519934


## Stereo geometry: B/H ratio and stereo angle

,pair,left_image,right_image,stereo_angle_deg,B_over_H,geometry_source
0,AB,A,B,26.282,0.467,incidence_angles_and_azimuth



SAVE_OUTPUTS = False, so CSV files were temporary and only displayed.
